In [1]:
!pip install sentencepiece sacrebleu
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.6 MB/s eta 0:00:00


In [2]:
!pip install IndicTransToolkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.3/546.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 45.7 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
from transformers import T5ForConditionalGeneration, AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import gc 
import json
import time
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

2026-03-01 05:03:21.703607: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772341402.049131      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772341402.138270      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772341402.998038      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772341402.998074      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772341402.998077      24 computation_placer.cc:177] computation placer alr

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)


# --- 1. Your Pre-trained Tokenwise DAE ---
class DAE(nn.Module):
    def __init__(self, d_in=768, d_latent=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, 512), nn.ReLU(),
            nn.Linear(512, d_latent)
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, 512), nn.ReLU(),
            nn.Linear(512, d_in)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z  # Return z (latent) for the next stage

# --- 2. The Contextual Encoder (Transformer) ---
class ContextualEncoder(nn.Module):
    def __init__(self, d_model=256, nhead=4, num_layers=4):
        super().__init__()
        # This layer allows tokens to "talk" to each other after the DAE isolated them
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
    def forward(self, x, mask=None):
        # x shape: (Batch, Seq_Len, 256)
        # We need to invert the mask for PyTorch Transformer (True = Ignore)
        # HuggingFace mask is (1 = Keep, 0 = Ignore), PyTorch expects opposite or float mask
        if mask is not None:
            # Create padding mask: (Batch, Seq_Len) where True means "pad" (ignore)
            padding_mask = (mask == 0) 
            return self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        return self.transformer_encoder(x)

# --- 3. The Full Pipeline Class ---
class CustomMTPipeline(nn.Module):
    def __init__(self, dae_path, t5_model_name="google/mt5-small"):
        super().__init__()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # A. IndicBERT (Backbone)
        self.indic = AutoModel.from_pretrained("ai4bharat/indic-bert")
        
        # B. DAE (Tokenwise Feature Extractor)
        self.dae = DAE(d_in=768, d_latent=256)
        # Load your Pre-trained weights
        checkpoint = torch.load(dae_path, map_location=self.device)
        
        # Check if it's a dictionary containing 'model_state_dict' or just weights
        if 'model_state_dict' in checkpoint:
            self.dae.load_state_dict(checkpoint['model_state_dict'])
        else:
            self.dae.load_state_dict(checkpoint)
        # FREEZE DAE (As you already trained it)
        for param in self.dae.parameters():
            param.requires_grad = False
            
        # C. Transformer Encoder (Learns Grammar/Context)
        # Input dim is 256 (output of DAE Encoder)
        self.context_encoder = ContextualEncoder(d_model=256, nhead=4, num_layers=4)
        
        # D. Projection to T5 (256 -> 512)
        self.projector = nn.Linear(256, 512)
        
        # E. T5 Decoder (Generates Output)
        self.t5 = T5ForConditionalGeneration.from_pretrained(t5_model_name)
        del self.t5.encoder # We don't use T5's native encoder

    def forward(self, src_ids, src_mask, tgt_ids):
        # 1. IndicBERT
        indic_out = self.indic(src_ids, attention_mask=src_mask).last_hidden_state # (B, T, 768)
        
        # 2. DAE Encoder (Tokenwise)
        _, latent_z = self.dae(indic_out) # (B, T, 256)
        
        # 3. Contextual Transformer Encoder
        # Crucial: Pass the mask so it doesn't attend to padding tokens
        context_out = self.context_encoder(latent_z, mask=src_mask) # (B, T, 256)
        
        # 4. Project to T5 Dimension
        decoder_input_embeds = self.projector(context_out) # (B, T, 512)
        
        # 5. T5 Decoder
        out = self.t5(
            encoder_outputs=(decoder_input_embeds,), 
            labels=tgt_ids,
            attention_mask=src_mask # Pass mask for cross-attention
        )
        return out.loss

DAE_PATH = '/kaggle/input/dae-model-epoch-7/pytorch/finalmodel/1/dae_model_final.pt'

# Tokenizers
indic_tok = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
t5_tok = AutoTokenizer.from_pretrained("google/mt5-small")


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [5]:
!pip install sacrebleu

# asm_Beng

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "as"  # The Decoder Language we are testing
DECODER_WEIGHTS = "/kaggle/input/decoder-asm-beng/pytorch/default/1/decoder_asm_Beng.pt"

# Shared Paths (Universal Encoder)
SHARED_ENC_PATH = "/kaggle/input/encoder-output/shared_context_encoder.pt"
SHARED_PROJ_PATH = "/kaggle/input/encoder-output/shared_projector.pt"

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Load separate tokenizer/model for evaluation to avoid conflict with main model
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize Pipeline
model = CustomMTPipeline(DAE_PATH).to(device)
print("Loading Model Weights...")
model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP (With Duplicate Check) ---
results = []
processed_languages = set() # Track processed languages here

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        # The source column is the one that is NOT the target
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        # --- FIX: Skip if we already tested this language ---
        if src_col in processed_languages:
            continue 
        
        # Mark as processed
        processed_languages.add(src_col)
        
        print(f"\nTesting Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        # Sample random rows
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
                _, latent_z = model.dae(indic_out)
                context_out = model.context_encoder(latent_z, mask=inputs.attention_mask)
                decoder_input_embeds = model.projector(context_out)
                
                # Wrap embeddings for T5
                encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
                gen_ids = model.t5.generate(
                    encoder_outputs=encoder_outputs_wrapped, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        # Standard ChrF++ (6 char, 2 word)
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR DECODER: {TARGET_LANG}")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    # Save results
    results_df.to_csv(f"final_avg_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

# With just mt5-small (asm)

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "as"  # The Decoder Language we are testing
# We use 'google/mt5-base' because standard 't5-base' does not support Indic scripts.
# If you have a fine-tuned T5 checkpoint, replace this string with your path (e.g., "/kaggle/input/my-finetuned-t5/")
BASELINE_MODEL_NAME = "google/mt5-small" 

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore (Kept identical for fair comparison) ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP BASELINE MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading Baseline Model: {BASELINE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(BASELINE_MODEL_NAME).to(device)
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP ---
results = []
processed_languages = set() 

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue 
        
        processed_languages.add(src_col)
        
        print(f"\n[Baseline] Testing Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            # Standard T5 Tokenization
            inputs = tokenizer(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                gen_ids = model.generate(
                    input_ids=inputs.input_ids, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            # Decode
            batch_preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR BASELINE (mT5-small)")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" BASELINE AVG BLEU:       {avg_bleu:.2f}")
    print(f" BASELINE AVG ChrF++:     {avg_chrf:.2f}")
    print(f" BASELINE AVG IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    results_df.to_csv(f"baseline_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

# Asm Indic Trans 2

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModel
from IndicTransToolkit import IndicProcessor
import torch.nn.functional as F
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)

# --- CONFIGURATION ---
TARGET_LANG = "as"  # Assamese
TARGET_LANG_CODE = "asm_Beng"

LANG_CODE_MAP = {
    "as": "asm_Beng", "bn": "ben_Beng", "hi": "hin_Deva", "en": "eng_Latn",
    "ta": "tam_Taml", "te": "tel_Telu", "mr": "mar_Deva", "gu": "guj_Gujr",
    "kn": "kan_Knda", "ml": "mal_Mlym", "pa": "pan_Guru", "or": "ory_Orya", "ur": "urd_Arab",
}

# SPEED OPTIMIZATION: Use smaller model for faster inference
# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"  # 1B params, slower
MODEL_NAME = "ai4bharat/indictrans2-indic-indic-dist-320M"  # 320M params, 3x faster

# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"


BATCH_SIZE = 32  # Increased from 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
    scores = []
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        with torch.no_grad():
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading IndicTrans2 model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, 
    trust_remote_code=True,
    torch_dtype=torch.float16,  # FP16 for speed
).to(device)
model.eval()

ip = IndicProcessor(inference=True)
print("IndicTrans2 model loaded successfully!")

# NEW: Load IndicBERT globally so it doesn't reload for every language
print("Loading IndicBERT evaluation model...")
eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
eval_model.eval()
print("All models loaded successfully!")

# --- 2. FIND FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found!")
    exit()

# --- 3. MAIN EVALUATION LOOP WITH PROGRESS BARS ---
results = []
processed_languages = set()

# Outer progress bar for languages
for file_path in tqdm(all_files, desc="Overall Progress", unit="language"):
    try:
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue
        
        processed_languages.add(src_col)
        
        src_lang_code = LANG_CODE_MAP.get(src_col, src_col)
        tgt_lang_code = LANG_CODE_MAP.get(TARGET_LANG, TARGET_LANG)
        
        print(f"\n{'='*60}")
        print(f"Testing: {src_col} ({src_lang_code}) → {TARGET_LANG} ({tgt_lang_code})")
        print(f"{'='*60}")
        
        # Load data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # Inference with inner progress bar
        predictions = []
        num_batches = (len(src_texts) + BATCH_SIZE - 1) // BATCH_SIZE
        
        print(f"Processing {len(src_texts)} sentences in {num_batches} batches...")
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            print(f"  Batch {i//BATCH_SIZE + 1}/{num_batches}...", end="\r")
                    
            # Preprocess
            batch = ip.preprocess_batch(batch_src, src_lang=src_lang_code, tgt_lang=tgt_lang_code)
            
            # Tokenize
            inputs = tokenizer(
                batch,
                truncation=True,
                padding="longest",
                return_tensors="pt",
                return_attention_mask=True
            ).to(device)
            
            # Generate (use_cache=False to avoid bug, num_beams=1 for speed)
            with torch.no_grad():
                generated_tokens = model.generate(
                    **inputs,
                    use_cache=False,
                    min_length=0,
                    max_length=256,
                    num_beams=1,
                    num_return_sequences=1,
                )
            
            # Decode (Removed deprecated as_target_tokenizer warning)
            batch_preds = tokenizer.batch_decode(
                generated_tokens, 
                skip_special_tokens=True, 
                clean_up_tokenization_spaces=True
            )
            
            # Post-process
            batch_preds = ip.postprocess_batch(batch_preds, lang=tgt_lang_code)
            predictions.extend(batch_preds)
            
            # Clean up GPU memory
            del inputs, generated_tokens
            torch.cuda.empty_cache()
        
        # Calculate Metrics
        print(f"\nCalculating metrics for {src_col}→{TARGET_LANG}...")
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
        print(f"✓ Results: BLEU={bleu.score:.2f} | ChrF++={chrf.score:.2f} | BERT={bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "Source Code": src_lang_code,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"\n✗ Error processing {os.path.basename(file_path)}: {e}")
        import traceback
        traceback.print_exc()

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print(f"FINAL AGGREGATE RESULTS FOR TARGET: {TARGET_LANG} ({tgt_lang_code})")
    print("="*70)
    print(results_df.round(2).to_string(index=False))
    print("-" * 70)
    
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*70)
    
    results_df.to_csv(f"indictrans2_results_{TARGET_LANG}.csv", index=False)
    print(f"✓ Results saved to indictrans2_results_{TARGET_LANG}.csv")
else:
    print("No results generated.")

# mal_Mlym

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "ml"  # The Decoder Language we are testing
DECODER_WEIGHTS = "/kaggle/input/decoder-mal-mlym/pytorch/default/1/decoder_mal_Mlym.pt"

# Shared Paths (Universal Encoder)
SHARED_ENC_PATH = "/kaggle/input/encoder-output/shared_context_encoder.pt"
SHARED_PROJ_PATH = "/kaggle/input/encoder-output/shared_projector.pt"

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Load separate tokenizer/model for evaluation to avoid conflict with main model
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize Pipeline
model = CustomMTPipeline(DAE_PATH).to(device)
print("Loading Model Weights...")
model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP (With Duplicate Check) ---
results = []
processed_languages = set() # Track processed languages here

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        # The source column is the one that is NOT the target
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        # --- FIX: Skip if we already tested this language ---
        if src_col in processed_languages:
            continue 
        
        # Mark as processed
        processed_languages.add(src_col)
        
        print(f"\nTesting Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        # Sample random rows
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
                _, latent_z = model.dae(indic_out)
                context_out = model.context_encoder(latent_z, mask=inputs.attention_mask)
                decoder_input_embeds = model.projector(context_out)
                
                # Wrap embeddings for T5
                encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
                gen_ids = model.t5.generate(
                    encoder_outputs=encoder_outputs_wrapped, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        # Standard ChrF++ (6 char, 2 word)
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR DECODER: {TARGET_LANG}")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    # Save results
    results_df.to_csv(f"final_avg_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

# With mt5-small(Malyalam)

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "ml"  # The Decoder Language we are testing
# We use 'google/mt5-base' because standard 't5-base' does not support Indic scripts.
# If you have a fine-tuned T5 checkpoint, replace this string with your path (e.g., "/kaggle/input/my-finetuned-t5/")
BASELINE_MODEL_NAME = "google/mt5-small" 

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore (Kept identical for fair comparison) ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP BASELINE MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading Baseline Model: {BASELINE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(BASELINE_MODEL_NAME).to(device)
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP ---
results = []
processed_languages = set() 

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue 
        
        processed_languages.add(src_col)
        
        print(f"\n[Baseline] Testing Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            # Standard T5 Tokenization
            inputs = tokenizer(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                gen_ids = model.generate(
                    input_ids=inputs.input_ids, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            # Decode
            batch_preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR BASELINE (mT5-small)")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" BASELINE AVG BLEU:       {avg_bleu:.2f}")
    print(f" BASELINE AVG ChrF++:     {avg_chrf:.2f}")
    print(f" BASELINE AVG IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    results_df.to_csv(f"baseline_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

# IndicTrans2 (Mlayalam)

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModel
from IndicTransToolkit import IndicProcessor
import torch.nn.functional as F
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)

# --- CONFIGURATION ---
TARGET_LANG = "ml"  # malyalam
TARGET_LANG_CODE = "mal_Mlym"

LANG_CODE_MAP = {
    "as": "asm_Beng", "bn": "ben_Beng", "hi": "hin_Deva", "en": "eng_Latn",
    "ta": "tam_Taml", "te": "tel_Telu", "mr": "mar_Deva", "gu": "guj_Gujr",
    "kn": "kan_Knda", "ml": "mal_Mlym", "pa": "pan_Guru", "or": "ory_Orya", "ur": "urd_Arab",
}

# SPEED OPTIMIZATION: Use smaller model for faster inference
# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"  # 1B params, slower
MODEL_NAME = "ai4bharat/indictrans2-indic-indic-dist-320M"  # 320M params, 3x faster

# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"


BATCH_SIZE = 32  # Increased from 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
    scores = []
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        with torch.no_grad():
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading IndicTrans2 model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, 
    trust_remote_code=True,
    torch_dtype=torch.float16,  # FP16 for speed
).to(device)
model.eval()

ip = IndicProcessor(inference=True)
print("IndicTrans2 model loaded successfully!")

# NEW: Load IndicBERT globally so it doesn't reload for every language
print("Loading IndicBERT evaluation model...")
eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
eval_model.eval()
print("All models loaded successfully!")

# --- 2. FIND FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found!")
    exit()

# --- 3. MAIN EVALUATION LOOP WITH PROGRESS BARS ---
results = []
processed_languages = set()

# Outer progress bar for languages
for file_path in tqdm(all_files, desc="Overall Progress", unit="language"):
    try:
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue
        
        processed_languages.add(src_col)
        
        src_lang_code = LANG_CODE_MAP.get(src_col, src_col)
        tgt_lang_code = LANG_CODE_MAP.get(TARGET_LANG, TARGET_LANG)
        
        print(f"\n{'='*60}")
        print(f"Testing: {src_col} ({src_lang_code}) → {TARGET_LANG} ({tgt_lang_code})")
        print(f"{'='*60}")
        
        # Load data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # Inference with inner progress bar
        predictions = []
        num_batches = (len(src_texts) + BATCH_SIZE - 1) // BATCH_SIZE
        
        print(f"Processing {len(src_texts)} sentences in {num_batches} batches...")
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            print(f"  Batch {i//BATCH_SIZE + 1}/{num_batches}...", end="\r")
                    
            # Preprocess
            batch = ip.preprocess_batch(batch_src, src_lang=src_lang_code, tgt_lang=tgt_lang_code)
            
            # Tokenize
            inputs = tokenizer(
                batch,
                truncation=True,
                padding="longest",
                return_tensors="pt",
                return_attention_mask=True
            ).to(device)
            
            # Generate (use_cache=False to avoid bug, num_beams=1 for speed)
            with torch.no_grad():
                generated_tokens = model.generate(
                    **inputs,
                    use_cache=False,
                    min_length=0,
                    max_length=256,
                    num_beams=1,
                    num_return_sequences=1,
                )
            
            # Decode (Removed deprecated as_target_tokenizer warning)
            batch_preds = tokenizer.batch_decode(
                generated_tokens, 
                skip_special_tokens=True, 
                clean_up_tokenization_spaces=True
            )
            
            # Post-process
            batch_preds = ip.postprocess_batch(batch_preds, lang=tgt_lang_code)
            predictions.extend(batch_preds)
            
            # Clean up GPU memory
            del inputs, generated_tokens
            torch.cuda.empty_cache()
        
        # Calculate Metrics
        print(f"\nCalculating metrics for {src_col}→{TARGET_LANG}...")
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
        print(f"✓ Results: BLEU={bleu.score:.2f} | ChrF++={chrf.score:.2f} | BERT={bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "Source Code": src_lang_code,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"\n✗ Error processing {os.path.basename(file_path)}: {e}")
        import traceback
        traceback.print_exc()

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print(f"FINAL AGGREGATE RESULTS FOR TARGET: {TARGET_LANG} ({tgt_lang_code})")
    print("="*70)
    print(results_df.round(2).to_string(index=False))
    print("-" * 70)
    
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*70)
    
    results_df.to_csv(f"indictrans2_results_{TARGET_LANG}.csv", index=False)
    print(f"✓ Results saved to indictrans2_results_{TARGET_LANG}.csv")
else:
    print("No results generated.")

# mar_Deva

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "mr"  # The Decoder Language we are testing
DECODER_WEIGHTS = "/kaggle/input/decoder-mar/pytorch/default/1/decoder_mar_Deva.pt"

# Shared Paths (Universal Encoder)
SHARED_ENC_PATH = "/kaggle/input/encoder-output/shared_context_encoder.pt"
SHARED_PROJ_PATH = "/kaggle/input/encoder-output/shared_projector.pt"

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Load separate tokenizer/model for evaluation to avoid conflict with main model
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize Pipeline
model = CustomMTPipeline(DAE_PATH).to(device)
print("Loading Model Weights...")
model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP (With Duplicate Check) ---
results = []
processed_languages = set() # Track processed languages here

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        # The source column is the one that is NOT the target
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        # --- FIX: Skip if we already tested this language ---
        if src_col in processed_languages:
            continue 
        
        # Mark as processed
        processed_languages.add(src_col)
        
        print(f"\nTesting Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        # Sample random rows
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
                _, latent_z = model.dae(indic_out)
                context_out = model.context_encoder(latent_z, mask=inputs.attention_mask)
                decoder_input_embeds = model.projector(context_out)
                
                # Wrap embeddings for T5
                encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
                gen_ids = model.t5.generate(
                    encoder_outputs=encoder_outputs_wrapped, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        # Standard ChrF++ (6 char, 2 word)
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR DECODER: {TARGET_LANG}")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    # Save results
    results_df.to_csv(f"final_avg_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

# With mt5-small (marathi)

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "mr"  # The Decoder Language we are testing
# We use 'google/mt5-base' because standard 't5-base' does not support Indic scripts.
# If you have a fine-tuned T5 checkpoint, replace this string with your path (e.g., "/kaggle/input/my-finetuned-t5/")
BASELINE_MODEL_NAME = "google/mt5-small" 

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore (Kept identical for fair comparison) ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP BASELINE MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading Baseline Model: {BASELINE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(BASELINE_MODEL_NAME).to(device)
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP ---
results = []
processed_languages = set() 

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue 
        
        processed_languages.add(src_col)
        
        print(f"\n[Baseline] Testing Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            # Standard T5 Tokenization
            inputs = tokenizer(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                gen_ids = model.generate(
                    input_ids=inputs.input_ids, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            # Decode
            batch_preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR BASELINE (mT5-small)")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" BASELINE AVG BLEU:       {avg_bleu:.2f}")
    print(f" BASELINE AVG ChrF++:     {avg_chrf:.2f}")
    print(f" BASELINE AVG IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    results_df.to_csv(f"baseline_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

# IndicTrans2(marathi)

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModel
from IndicTransToolkit import IndicProcessor
import torch.nn.functional as F
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)

# --- CONFIGURATION ---
TARGET_LANG = "ma"  # malyalam
TARGET_LANG_CODE = "mar_Deva"

LANG_CODE_MAP = {
    "as": "asm_Beng", "bn": "ben_Beng", "hi": "hin_Deva", "en": "eng_Latn",
    "ta": "tam_Taml", "te": "tel_Telu", "mr": "mar_Deva", "gu": "guj_Gujr",
    "kn": "kan_Knda", "ml": "mal_Mlym", "pa": "pan_Guru", "or": "ory_Orya", "ur": "urd_Arab",
}

# SPEED OPTIMIZATION: Use smaller model for faster inference
# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"  # 1B params, slower
MODEL_NAME = "ai4bharat/indictrans2-indic-indic-dist-320M"  # 320M params, 3x faster

# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"


BATCH_SIZE = 32  # Increased from 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
    scores = []
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        with torch.no_grad():
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading IndicTrans2 model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, 
    trust_remote_code=True,
    torch_dtype=torch.float16,  # FP16 for speed
).to(device)
model.eval()

ip = IndicProcessor(inference=True)
print("IndicTrans2 model loaded successfully!")

# NEW: Load IndicBERT globally so it doesn't reload for every language
print("Loading IndicBERT evaluation model...")
eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
eval_model.eval()
print("All models loaded successfully!")

# --- 2. FIND FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found!")
    exit()

# --- 3. MAIN EVALUATION LOOP WITH PROGRESS BARS ---
results = []
processed_languages = set()

# Outer progress bar for languages
for file_path in tqdm(all_files, desc="Overall Progress", unit="language"):
    try:
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue
        
        processed_languages.add(src_col)
        
        src_lang_code = LANG_CODE_MAP.get(src_col, src_col)
        tgt_lang_code = LANG_CODE_MAP.get(TARGET_LANG, TARGET_LANG)
        
        print(f"\n{'='*60}")
        print(f"Testing: {src_col} ({src_lang_code}) → {TARGET_LANG} ({tgt_lang_code})")
        print(f"{'='*60}")
        
        # Load data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # Inference with inner progress bar
        predictions = []
        num_batches = (len(src_texts) + BATCH_SIZE - 1) // BATCH_SIZE
        
        print(f"Processing {len(src_texts)} sentences in {num_batches} batches...")
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            print(f"  Batch {i//BATCH_SIZE + 1}/{num_batches}...", end="\r")
                    
            # Preprocess
            batch = ip.preprocess_batch(batch_src, src_lang=src_lang_code, tgt_lang=tgt_lang_code)
            
            # Tokenize
            inputs = tokenizer(
                batch,
                truncation=True,
                padding="longest",
                return_tensors="pt",
                return_attention_mask=True
            ).to(device)
            
            # Generate (use_cache=False to avoid bug, num_beams=1 for speed)
            with torch.no_grad():
                generated_tokens = model.generate(
                    **inputs,
                    use_cache=False,
                    min_length=0,
                    max_length=256,
                    num_beams=1,
                    num_return_sequences=1,
                )
            
            # Decode (Removed deprecated as_target_tokenizer warning)
            batch_preds = tokenizer.batch_decode(
                generated_tokens, 
                skip_special_tokens=True, 
                clean_up_tokenization_spaces=True
            )
            
            # Post-process
            batch_preds = ip.postprocess_batch(batch_preds, lang=tgt_lang_code)
            predictions.extend(batch_preds)
            
            # Clean up GPU memory
            del inputs, generated_tokens
            torch.cuda.empty_cache()
        
        # Calculate Metrics
        print(f"\nCalculating metrics for {src_col}→{TARGET_LANG}...")
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
        print(f"✓ Results: BLEU={bleu.score:.2f} | ChrF++={chrf.score:.2f} | BERT={bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "Source Code": src_lang_code,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"\n✗ Error processing {os.path.basename(file_path)}: {e}")
        import traceback
        traceback.print_exc()

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print(f"FINAL AGGREGATE RESULTS FOR TARGET: {TARGET_LANG} ({tgt_lang_code})")
    print("="*70)
    print(results_df.round(2).to_string(index=False))
    print("-" * 70)
    
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*70)
    
    results_df.to_csv(f"indictrans2_results_{TARGET_LANG}.csv", index=False)
    print(f"✓ Results saved to indictrans2_results_{TARGET_LANG}.csv")
else:
    print("No results generated.")

# guj_Gujr

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "gu"  # The Decoder Language we are testing
DECODER_WEIGHTS = "/kaggle/input/decoder-gur-and-orya/pytorch/default/1/decoder_guj_Gujr.pt"

# Shared Paths (Universal Encoder)
SHARED_ENC_PATH = "/kaggle/input/encoder-output/shared_context_encoder.pt"
SHARED_PROJ_PATH = "/kaggle/input/encoder-output/shared_projector.pt"

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Load separate tokenizer/model for evaluation to avoid conflict with main model
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize Pipeline
model = CustomMTPipeline(DAE_PATH).to(device)
print("Loading Model Weights...")
model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP (With Duplicate Check) ---
results = []
processed_languages = set() # Track processed languages here

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        # The source column is the one that is NOT the target
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        # --- FIX: Skip if we already tested this language ---
        if src_col in processed_languages:
            continue 
        
        # Mark as processed
        processed_languages.add(src_col)
        
        print(f"\nTesting Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        # Sample random rows
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
                _, latent_z = model.dae(indic_out)
                context_out = model.context_encoder(latent_z, mask=inputs.attention_mask)
                decoder_input_embeds = model.projector(context_out)
                
                # Wrap embeddings for T5
                encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
                gen_ids = model.t5.generate(
                    encoder_outputs=encoder_outputs_wrapped, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        # Standard ChrF++ (6 char, 2 word)
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR DECODER: {TARGET_LANG}")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    # Save results
    results_df.to_csv(f"final_avg_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

# With mt5-small (guj)

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "gu"  # The Decoder Language we are testing
# We use 'google/mt5-base' because standard 't5-base' does not support Indic scripts.
# If you have a fine-tuned T5 checkpoint, replace this string with your path (e.g., "/kaggle/input/my-finetuned-t5/")
BASELINE_MODEL_NAME = "google/mt5-small" 

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore (Kept identical for fair comparison) ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP BASELINE MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading Baseline Model: {BASELINE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(BASELINE_MODEL_NAME).to(device)
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP ---
results = []
processed_languages = set() 

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue 
        
        processed_languages.add(src_col)
        
        print(f"\n[Baseline] Testing Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            # Standard T5 Tokenization
            inputs = tokenizer(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                gen_ids = model.generate(
                    input_ids=inputs.input_ids, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            # Decode
            batch_preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR BASELINE (mT5-small)")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" BASELINE AVG BLEU:       {avg_bleu:.2f}")
    print(f" BASELINE AVG ChrF++:     {avg_chrf:.2f}")
    print(f" BASELINE AVG IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    results_df.to_csv(f"baseline_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

# IndicTrans2 (guj)

In [ ]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModel
from IndicTransToolkit import IndicProcessor
import torch.nn.functional as F
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)

# --- CONFIGURATION ---
TARGET_LANG = "gu"  # gujrati 
TARGET_LANG_CODE = "guj_Gujr"

LANG_CODE_MAP = {
    "as": "asm_Beng", "bn": "ben_Beng", "hi": "hin_Deva", "en": "eng_Latn",
    "ta": "tam_Taml", "te": "tel_Telu", "mr": "mar_Deva", "gu": "guj_Gujr",
    "kn": "kan_Knda", "ml": "mal_Mlym", "pa": "pan_Guru", "or": "ory_Orya", "ur": "urd_Arab",
}

# SPEED OPTIMIZATION: Use smaller model for faster inference
# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"  # 1B params, slower
MODEL_NAME = "ai4bharat/indictrans2-indic-indic-dist-320M"  # 320M params, 3x faster

MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"


BATCH_SIZE = 32  # Increased from 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
    scores = []
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        with torch.no_grad():
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading IndicTrans2 model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, 
    trust_remote_code=True,
    torch_dtype=torch.float16,  # FP16 for speed
).to(device)
model.eval()

ip = IndicProcessor(inference=True)
print("IndicTrans2 model loaded successfully!")

# NEW: Load IndicBERT globally so it doesn't reload for every language
print("Loading IndicBERT evaluation model...")
eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
eval_model.eval()
print("All models loaded successfully!")

# --- 2. FIND FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found!")
    exit()

# --- 3. MAIN EVALUATION LOOP WITH PROGRESS BARS ---
results = []
processed_languages = set()

# Outer progress bar for languages
for file_path in tqdm(all_files, desc="Overall Progress", unit="language"):
    try:
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue
        
        processed_languages.add(src_col)
        
        src_lang_code = LANG_CODE_MAP.get(src_col, src_col)
        tgt_lang_code = LANG_CODE_MAP.get(TARGET_LANG, TARGET_LANG)
        
        print(f"\n{'='*60}")
        print(f"Testing: {src_col} ({src_lang_code}) → {TARGET_LANG} ({tgt_lang_code})")
        print(f"{'='*60}")
        
        # Load data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # Inference with inner progress bar
        predictions = []
        num_batches = (len(src_texts) + BATCH_SIZE - 1) // BATCH_SIZE
        
        print(f"Processing {len(src_texts)} sentences in {num_batches} batches...")
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            print(f"  Batch {i//BATCH_SIZE + 1}/{num_batches}...", end="\r")
                    
            # Preprocess
            batch = ip.preprocess_batch(batch_src, src_lang=src_lang_code, tgt_lang=tgt_lang_code)
            
            # Tokenize
            inputs = tokenizer(
                batch,
                truncation=True,
                padding="longest",
                return_tensors="pt",
                return_attention_mask=True
            ).to(device)
            
            # Generate (use_cache=False to avoid bug, num_beams=1 for speed)
            with torch.no_grad():
                generated_tokens = model.generate(
                    **inputs,
                    use_cache=False,
                    min_length=0,
                    max_length=256,
                    num_beams=1,
                    num_return_sequences=1,
                )
            
            # Decode (Removed deprecated as_target_tokenizer warning)
            batch_preds = tokenizer.batch_decode(
                generated_tokens, 
                skip_special_tokens=True, 
                clean_up_tokenization_spaces=True
            )
            
            # Post-process
            batch_preds = ip.postprocess_batch(batch_preds, lang=tgt_lang_code)
            predictions.extend(batch_preds)
            
            # Clean up GPU memory
            del inputs, generated_tokens
            torch.cuda.empty_cache()
        
        # Calculate Metrics
        print(f"\nCalculating metrics for {src_col}→{TARGET_LANG}...")
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
        print(f"✓ Results: BLEU={bleu.score:.2f} | ChrF++={chrf.score:.2f} | BERT={bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "Source Code": src_lang_code,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"\n✗ Error processing {os.path.basename(file_path)}: {e}")
        import traceback
        traceback.print_exc()

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print(f"FINAL AGGREGATE RESULTS FOR TARGET: {TARGET_LANG} ({tgt_lang_code})")
    print("="*70)
    print(results_df.round(2).to_string(index=False))
    print("-" * 70)
    
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*70)
    
    results_df.to_csv(f"indictrans2_results_{TARGET_LANG}.csv", index=False)
    print(f"✓ Results saved to indictrans2_results_{TARGET_LANG}.csv")
else:
    print("No results generated.")

# tam_Taml

In [18]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "ta"  # The Decoder Language we are testing
DECODER_WEIGHTS = "/kaggle/input/decoder-tam-taml/pytorch/default/1/decoder_tam_Taml.pt"

# Shared Paths (Universal Encoder)
SHARED_ENC_PATH = "/kaggle/input/encoder-output/shared_context_encoder.pt"
SHARED_PROJ_PATH = "/kaggle/input/encoder-output/shared_projector.pt"

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Load separate tokenizer/model for evaluation to avoid conflict with main model
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize Pipeline
model = CustomMTPipeline(DAE_PATH).to(device)
print("Loading Model Weights...")
model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP (With Duplicate Check) ---
results = []
processed_languages = set() # Track processed languages here

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        # The source column is the one that is NOT the target
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        # --- FIX: Skip if we already tested this language ---
        if src_col in processed_languages:
            continue 
        
        # Mark as processed
        processed_languages.add(src_col)
        
        print(f"\nTesting Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        # Sample random rows
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
                _, latent_z = model.dae(indic_out)
                context_out = model.context_encoder(latent_z, mask=inputs.attention_mask)
                decoder_input_embeds = model.projector(context_out)
                
                # Wrap embeddings for T5
                encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
                gen_ids = model.t5.generate(
                    encoder_outputs=encoder_outputs_wrapped, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        # Standard ChrF++ (6 char, 2 word)
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR DECODER: {TARGET_LANG}")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    # Save results
    results_df.to_csv(f"final_avg_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

Using device: cuda


pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

You are using a model of type mt5 to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


model.safetensors:   0%|          | 0.00/135M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

...

   >> BLEU: 2.23 | ChrF++: 30.02 | BERT: 0.9083

Testing Pair: or -> ta


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 2.12 | ChrF++: 29.62 | BERT: 0.9089

Testing Pair: gu -> ta


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 2.21 | ChrF++: 29.79 | BERT: 0.9141

FINAL AGGREGATE RESULTS FOR DECODER: ta
Source Language  BLEU  ChrF++  IndicBERT
             as  2.04   29.81       0.91
             ml  1.66   27.99       0.90
             mr  2.18   29.67       0.91
             bn  2.23   30.02       0.91
             or  2.12   29.62       0.91
             gu  2.21   29.79       0.91
------------------------------------------------------------
 GLOBAL AVERAGE BLEU:       2.08
 GLOBAL AVERAGE ChrF++:     29.48
 GLOBAL AVERAGE IndicBERT:  0.9080


# With mt5-small (tamil)

In [19]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "ta"  # The Decoder Language we are testing
# We use 'google/mt5-base' because standard 't5-base' does not support Indic scripts.
# If you have a fine-tuned T5 checkpoint, replace this string with your path (e.g., "/kaggle/input/my-finetuned-t5/")
BASELINE_MODEL_NAME = "google/mt5-small" 

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore (Kept identical for fair comparison) ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP BASELINE MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading Baseline Model: {BASELINE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(BASELINE_MODEL_NAME).to(device)
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP ---
results = []
processed_languages = set() 

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue 
        
        processed_languages.add(src_col)
        
        print(f"\n[Baseline] Testing Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            # Standard T5 Tokenization
            inputs = tokenizer(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                gen_ids = model.generate(
                    input_ids=inputs.input_ids, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            # Decode
            batch_preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR BASELINE (mT5-small)")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" BASELINE AVG BLEU:       {avg_bleu:.2f}")
    print(f" BASELINE AVG ChrF++:     {avg_chrf:.2f}")
    print(f" BASELINE AVG IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    results_df.to_csv(f"baseline_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

Using device: cuda
Loading Baseline Model: google/mt5-small...


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Searching for pairs ending in ..._to_ta.csv

[Baseline] Testing Pair: as -> ta


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 0.00 | ChrF++: 0.05 | BERT: 0.5190

[Baseline] Testing Pair: ml -> ta


...

   >> BLEU: 0.00 | ChrF++: 0.05 | BERT: 0.5011

[Baseline] Testing Pair: or -> ta


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 0.00 | ChrF++: 0.20 | BERT: 0.5333

[Baseline] Testing Pair: gu -> ta


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 0.03 | ChrF++: 0.73 | BERT: 0.4923

FINAL AGGREGATE RESULTS FOR BASELINE (mT5-small)
Source Language  BLEU  ChrF++  IndicBERT
             as  0.00    0.05       0.52
             ml  0.01    0.91       0.52
             mr  0.00    0.79       0.53
             bn  0.00    0.05       0.50
             or  0.00    0.20       0.53
             gu  0.03    0.73       0.49
------------------------------------------------------------
 BASELINE AVG BLEU:       0.01
 BASELINE AVG ChrF++:     0.46
 BASELINE AVG IndicBERT:  0.5166


# IndicTrans2(tamil)

In [20]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModel
from IndicTransToolkit import IndicProcessor
import torch.nn.functional as F
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)

# --- CONFIGURATION ---
TARGET_LANG = "ta"  # tamil
TARGET_LANG_CODE = "tam_Taml"

LANG_CODE_MAP = {
    "as": "asm_Beng", "bn": "ben_Beng", "hi": "hin_Deva", "en": "eng_Latn",
    "ta": "tam_Taml", "te": "tel_Telu", "mr": "mar_Deva", "gu": "guj_Gujr",
    "kn": "kan_Knda", "ml": "mal_Mlym", "pa": "pan_Guru", "or": "ory_Orya", "ur": "urd_Arab",
}

# SPEED OPTIMIZATION: Use smaller model for faster inference
# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"  # 1B params, slower
MODEL_NAME = "ai4bharat/indictrans2-indic-indic-dist-320M"  # 320M params, 3x faster

MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"


BATCH_SIZE = 32  # Increased from 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
    scores = []
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        with torch.no_grad():
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading IndicTrans2 model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, 
    trust_remote_code=True,
    torch_dtype=torch.float16,  # FP16 for speed
).to(device)
model.eval()

ip = IndicProcessor(inference=True)
print("IndicTrans2 model loaded successfully!")

# NEW: Load IndicBERT globally so it doesn't reload for every language
print("Loading IndicBERT evaluation model...")
eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
eval_model.eval()
print("All models loaded successfully!")

# --- 2. FIND FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found!")
    exit()

# --- 3. MAIN EVALUATION LOOP WITH PROGRESS BARS ---
results = []
processed_languages = set()

# Outer progress bar for languages
for file_path in tqdm(all_files, desc="Overall Progress", unit="language"):
    try:
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue
        
        processed_languages.add(src_col)
        
        src_lang_code = LANG_CODE_MAP.get(src_col, src_col)
        tgt_lang_code = LANG_CODE_MAP.get(TARGET_LANG, TARGET_LANG)
        
        print(f"\n{'='*60}")
        print(f"Testing: {src_col} ({src_lang_code}) → {TARGET_LANG} ({tgt_lang_code})")
        print(f"{'='*60}")
        
        # Load data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # Inference with inner progress bar
        predictions = []
        num_batches = (len(src_texts) + BATCH_SIZE - 1) // BATCH_SIZE
        
        print(f"Processing {len(src_texts)} sentences in {num_batches} batches...")
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            print(f"  Batch {i//BATCH_SIZE + 1}/{num_batches}...", end="\r")
                    
            # Preprocess
            batch = ip.preprocess_batch(batch_src, src_lang=src_lang_code, tgt_lang=tgt_lang_code)
            
            # Tokenize
            inputs = tokenizer(
                batch,
                truncation=True,
                padding="longest",
                return_tensors="pt",
                return_attention_mask=True
            ).to(device)
            
            # Generate (use_cache=False to avoid bug, num_beams=1 for speed)
            with torch.no_grad():
                generated_tokens = model.generate(
                    **inputs,
                    use_cache=False,
                    min_length=0,
                    max_length=256,
                    num_beams=1,
                    num_return_sequences=1,
                )
            
            # Decode (Removed deprecated as_target_tokenizer warning)
            batch_preds = tokenizer.batch_decode(
                generated_tokens, 
                skip_special_tokens=True, 
                clean_up_tokenization_spaces=True
            )
            
            # Post-process
            batch_preds = ip.postprocess_batch(batch_preds, lang=tgt_lang_code)
            predictions.extend(batch_preds)
            
            # Clean up GPU memory
            del inputs, generated_tokens
            torch.cuda.empty_cache()
        
        # Calculate Metrics
        print(f"\nCalculating metrics for {src_col}→{TARGET_LANG}...")
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
        print(f"✓ Results: BLEU={bleu.score:.2f} | ChrF++={chrf.score:.2f} | BERT={bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "Source Code": src_lang_code,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"\n✗ Error processing {os.path.basename(file_path)}: {e}")
        import traceback
        traceback.print_exc()

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print(f"FINAL AGGREGATE RESULTS FOR TARGET: {TARGET_LANG} ({tgt_lang_code})")
    print("="*70)
    print(results_df.round(2).to_string(index=False))
    print("-" * 70)
    
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*70)
    
    results_df.to_csv(f"indictrans2_results_{TARGET_LANG}.csv", index=False)
    print(f"✓ Results saved to indictrans2_results_{TARGET_LANG}.csv")
else:
    print("No results generated.")

Using device: cuda
Loading IndicTrans2 model: ai4bharat/indictrans2-indic-indic-1B


tokenizer_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenization_indictrans.py:   0%|          | 0.00/8.04k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json:   0%|          | 0.00/3.39M [00:00<?, ?B/s]

...

✓ Results: BLEU=7.52 | ChrF++=43.69 | BERT=0.9290

Testing: or (ory_Orya) → ta (tam_Taml)
Processing 1024 sentences in 32 batches...
  Batch 32/32...
Calculating metrics for or→ta...


Overall Progress:  83%|████████▎ | 5/6 [2:24:13<29:34, 1774.79s/language]  

✓ Results: BLEU=9.93 | ChrF++=47.78 | BERT=0.9589

Testing: gu (guj_Gujr) → ta (tam_Taml)
Processing 1024 sentences in 32 batches...
  Batch 32/32...
Calculating metrics for gu→ta...


Overall Progress: 100%|██████████| 6/6 [2:44:36<00:00, 1646.11s/language]

✓ Results: BLEU=10.45 | ChrF++=47.43 | BERT=0.9604

FINAL AGGREGATE RESULTS FOR TARGET: ta (tam_Taml)
Source Language Source Code  BLEU  ChrF++  IndicBERT
             as    asm_Beng  9.76   46.85       0.95
             ml    mal_Mlym 10.81   48.19       0.96
             mr    mar_Deva 10.76   48.44       0.96
             bn    ben_Beng  7.52   43.69       0.93
             or    ory_Orya  9.93   47.78       0.96
             gu    guj_Gujr 10.45   47.43       0.96
----------------------------------------------------------------------
 GLOBAL AVERAGE BLEU:       9.87
 GLOBAL AVERAGE ChrF++:     47.06
 GLOBAL AVERAGE IndicBERT:  0.9544
✓ Results saved to indictrans2_results_ta.csv


# ory_Orya

In [21]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "or"  # The Decoder Language we are testing
DECODER_WEIGHTS = "/kaggle/input/decoder-gur-and-orya/pytorch/default/1/decoder_ory_Orya.pt"

# Shared Paths (Universal Encoder)
SHARED_ENC_PATH = "/kaggle/input/encoder-output/shared_context_encoder.pt"
SHARED_PROJ_PATH = "/kaggle/input/encoder-output/shared_projector.pt"

# Evaluation 
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Load separate tokenizer/model for evaluation to avoid conflict with main model
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize Pipeline
model = CustomMTPipeline(DAE_PATH).to(device)
print("Loading Model Weights...")
model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP (With Duplicate Check) ---
results = []
processed_languages = set() # Track processed languages here

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        # The source column is the one that is NOT the target
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        # --- FIX: Skip if we already tested this language ---
        if src_col in processed_languages:
            continue 
        
        # Mark as processed
        processed_languages.add(src_col)
        
        print(f"\nTesting Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        # Sample random rows
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
                _, latent_z = model.dae(indic_out)
                context_out = model.context_encoder(latent_z, mask=inputs.attention_mask)
                decoder_input_embeds = model.projector(context_out)
                
                # Wrap embeddings for T5
                encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
                gen_ids = model.t5.generate(
                    encoder_outputs=encoder_outputs_wrapped, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        # Standard ChrF++ (6 char, 2 word)
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR DECODER: {TARGET_LANG}")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    # Save results
    results_df.to_csv(f"final_avg_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

Using device: cuda


You are using a model of type mt5 to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


Loading Model Weights...
Searching for pairs ending in ..._to_or.csv

Testing Pair: as -> or


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 1.52 | ChrF++: 19.10 | BERT: 0.8351

Testing Pair: mr -> or


...

   >> BLEU: 1.11 | ChrF++: 15.67 | BERT: 0.8313

Testing Pair: bn -> or


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 1.30 | ChrF++: 18.65 | BERT: 0.8465

Testing Pair: ta -> or


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 1.13 | ChrF++: 15.40 | BERT: 0.8247

FINAL AGGREGATE RESULTS FOR DECODER: or
Source Language  BLEU  ChrF++  IndicBERT
             as  1.52   19.10       0.84
             mr  1.36   17.54       0.83
             gu  1.15   17.84       0.84
             ml  1.11   15.67       0.83
             bn  1.30   18.65       0.85
             ta  1.13   15.40       0.82
------------------------------------------------------------
 GLOBAL AVERAGE BLEU:       1.26
 GLOBAL AVERAGE ChrF++:     17.37
 GLOBAL AVERAGE IndicBERT:  0.8351


# With mt5-small (orya)

In [22]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "or"  # The Decoder Language we are testing
# We use 'google/mt5-base' because standard 't5-base' does not support Indic scripts.
# If you have a fine-tuned T5 checkpoint, replace this string with your path (e.g., "/kaggle/input/my-finetuned-t5/")
BASELINE_MODEL_NAME = "google/mt5-small" 

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore (Kept identical for fair comparison) ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP BASELINE MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading Baseline Model: {BASELINE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(BASELINE_MODEL_NAME).to(device)
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP ---
results = []
processed_languages = set() 

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue 
        
        processed_languages.add(src_col)
        
        print(f"\n[Baseline] Testing Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            # Standard T5 Tokenization
            inputs = tokenizer(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                gen_ids = model.generate(
                    input_ids=inputs.input_ids, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            # Decode
            batch_preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR BASELINE (mT5-small)")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" BASELINE AVG BLEU:       {avg_bleu:.2f}")
    print(f" BASELINE AVG ChrF++:     {avg_chrf:.2f}")
    print(f" BASELINE AVG IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    results_df.to_csv(f"baseline_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

Using device: cuda
Loading Baseline Model: google/mt5-small...


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Searching for pairs ending in ..._to_or.csv

[Baseline] Testing Pair: as -> or


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 0.00 | ChrF++: 0.05 | BERT: 0.5151

[Baseline] Testing Pair: mr -> or


...

   >> BLEU: 0.00 | ChrF++: 0.11 | BERT: 0.5037

[Baseline] Testing Pair: bn -> or


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 0.00 | ChrF++: 0.12 | BERT: 0.4950

[Baseline] Testing Pair: ta -> or


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 0.00 | ChrF++: 0.11 | BERT: 0.5043

FINAL AGGREGATE RESULTS FOR BASELINE (mT5-small)
Source Language  BLEU  ChrF++  IndicBERT
             as   0.0    0.05       0.52
             mr   0.0    0.10       0.51
             gu   0.0    0.09       0.48
             ml   0.0    0.11       0.50
             bn   0.0    0.12       0.50
             ta   0.0    0.11       0.50
------------------------------------------------------------
 BASELINE AVG BLEU:       0.00
 BASELINE AVG ChrF++:     0.10
 BASELINE AVG IndicBERT:  0.5019


# IndicTrans2 (orya)

In [23]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModel
from IndicTransToolkit import IndicProcessor
import torch.nn.functional as F
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)

# --- CONFIGURATION ---
TARGET_LANG = "or"  #ory_Orya
TARGET_LANG_CODE = "ory_Orya"

LANG_CODE_MAP = {
    "as": "asm_Beng", "bn": "ben_Beng", "hi": "hin_Deva", "en": "eng_Latn",
    "ta": "tam_Taml", "te": "tel_Telu", "mr": "mar_Deva", "gu": "guj_Gujr",
    "kn": "kan_Knda", "ml": "mal_Mlym", "pa": "pan_Guru", "or": "ory_Orya", "ur": "urd_Arab",
}

# SPEED OPTIMIZATION: Use smaller model for faster inference
# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"  # 1B params, slower
MODEL_NAME = "ai4bharat/indictrans2-indic-indic-dist-320M"  # 320M params, 3x faster

# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"


BATCH_SIZE = 32  # Increased from 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
    scores = []
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        with torch.no_grad():
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading IndicTrans2 model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, 
    trust_remote_code=True,
    torch_dtype=torch.float16,  # FP16 for speed
).to(device)
model.eval()

ip = IndicProcessor(inference=True)
print("IndicTrans2 model loaded successfully!")

# NEW: Load IndicBERT globally so it doesn't reload for every language
print("Loading IndicBERT evaluation model...")
eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
eval_model.eval()
print("All models loaded successfully!")

# --- 2. FIND FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found!")
    exit()

# --- 3. MAIN EVALUATION LOOP WITH PROGRESS BARS ---
results = []
processed_languages = set()

# Outer progress bar for languages
for file_path in tqdm(all_files, desc="Overall Progress", unit="language"):
    try:
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue
        
        processed_languages.add(src_col)
        
        src_lang_code = LANG_CODE_MAP.get(src_col, src_col)
        tgt_lang_code = LANG_CODE_MAP.get(TARGET_LANG, TARGET_LANG)
        
        print(f"\n{'='*60}")
        print(f"Testing: {src_col} ({src_lang_code}) → {TARGET_LANG} ({tgt_lang_code})")
        print(f"{'='*60}")
        
        # Load data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # Inference with inner progress bar
        predictions = []
        num_batches = (len(src_texts) + BATCH_SIZE - 1) // BATCH_SIZE
        
        print(f"Processing {len(src_texts)} sentences in {num_batches} batches...")
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            print(f"  Batch {i//BATCH_SIZE + 1}/{num_batches}...", end="\r")
                    
            # Preprocess
            batch = ip.preprocess_batch(batch_src, src_lang=src_lang_code, tgt_lang=tgt_lang_code)
            
            # Tokenize
            inputs = tokenizer(
                batch,
                truncation=True,
                padding="longest",
                return_tensors="pt",
                return_attention_mask=True
            ).to(device)
            
            # Generate (use_cache=False to avoid bug, num_beams=1 for speed)
            with torch.no_grad():
                generated_tokens = model.generate(
                    **inputs,
                    use_cache=False,
                    min_length=0,
                    max_length=256,
                    num_beams=1,
                    num_return_sequences=1,
                )
            
            # Decode (Removed deprecated as_target_tokenizer warning)
            batch_preds = tokenizer.batch_decode(
                generated_tokens, 
                skip_special_tokens=True, 
                clean_up_tokenization_spaces=True
            )
            
            # Post-process
            batch_preds = ip.postprocess_batch(batch_preds, lang=tgt_lang_code)
            predictions.extend(batch_preds)
            
            # Clean up GPU memory
            del inputs, generated_tokens
            torch.cuda.empty_cache()
        
        # Calculate Metrics
        print(f"\nCalculating metrics for {src_col}→{TARGET_LANG}...")
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
        print(f"✓ Results: BLEU={bleu.score:.2f} | ChrF++={chrf.score:.2f} | BERT={bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "Source Code": src_lang_code,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"\n✗ Error processing {os.path.basename(file_path)}: {e}")
        import traceback
        traceback.print_exc()

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print(f"FINAL AGGREGATE RESULTS FOR TARGET: {TARGET_LANG} ({tgt_lang_code})")
    print("="*70)
    print(results_df.round(2).to_string(index=False))
    print("-" * 70)
    
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*70)
    
    results_df.to_csv(f"indictrans2_results_{TARGET_LANG}.csv", index=False)
    print(f"✓ Results saved to indictrans2_results_{TARGET_LANG}.csv")
else:
    print("No results generated.")

Using device: cuda
Loading IndicTrans2 model: ai4bharat/indictrans2-indic-indic-dist-320M


tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

tokenization_indictrans.py:   0%|          | 0.00/8.04k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-dist-320M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json:   0%|          | 0.00/3.39M [00:00<?, ?B/s]

...

✓ Results: BLEU=7.01 | ChrF++=38.96 | BERT=0.9397

Testing: bn (ben_Beng) → or (ory_Orya)
Processing 1024 sentences in 32 batches...
  Batch 32/32...
Calculating metrics for bn→or...


Overall Progress:  83%|████████▎ | 5/6 [49:17<11:31, 691.50s/language]

✓ Results: BLEU=5.11 | ChrF++=36.40 | BERT=0.9011

Testing: ta (tam_Taml) → or (ory_Orya)
Processing 1024 sentences in 32 batches...
  Batch 32/32...
Calculating metrics for ta→or...


Overall Progress: 100%|██████████| 6/6 [54:40<00:00, 546.76s/language]

✓ Results: BLEU=7.23 | ChrF++=38.26 | BERT=0.9418

FINAL AGGREGATE RESULTS FOR TARGET: or (ory_Orya)
Source Language Source Code  BLEU  ChrF++  IndicBERT
             as    asm_Beng  6.89   40.02       0.93
             mr    mar_Deva  7.52   39.90       0.94
             gu    guj_Gujr  6.88   39.74       0.94
             ml    mal_Mlym  7.01   38.96       0.94
             bn    ben_Beng  5.11   36.40       0.90
             ta    tam_Taml  7.23   38.26       0.94
----------------------------------------------------------------------
 GLOBAL AVERAGE BLEU:       6.77
 GLOBAL AVERAGE ChrF++:     38.88
 GLOBAL AVERAGE IndicBERT:  0.9316
✓ Results saved to indictrans2_results_or.csv


# ben_Beng

In [24]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "bn"  # The Decoder Language we are testing
DECODER_WEIGHTS = "/kaggle/input/decoder-bengali/pytorch/default/1/decoder_ben_Beng.pt"

# Shared Paths (Universal Encoder)
SHARED_ENC_PATH = "/kaggle/input/encoder-output/shared_context_encoder.pt"
SHARED_PROJ_PATH = "/kaggle/input/encoder-output/shared_projector.pt"

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Load separate tokenizer/model for evaluation to avoid conflict with main model
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize Pipeline
model = CustomMTPipeline(DAE_PATH).to(device)
print("Loading Model Weights...")
model.context_encoder.load_state_dict(torch.load(SHARED_ENC_PATH, map_location=device))
model.projector.load_state_dict(torch.load(SHARED_PROJ_PATH, map_location=device))
model.t5.load_state_dict(torch.load(DECODER_WEIGHTS, map_location=device))
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP (With Duplicate Check) ---
results = []
processed_languages = set() # Track processed languages here

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        # The source column is the one that is NOT the target
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        # --- FIX: Skip if we already tested this language ---
        if src_col in processed_languages:
            continue 
        
        # Mark as processed
        processed_languages.add(src_col)
        
        print(f"\nTesting Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        # Sample random rows
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            inputs = indic_tok(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                indic_out = model.indic(inputs.input_ids, attention_mask=inputs.attention_mask).last_hidden_state
                _, latent_z = model.dae(indic_out)
                context_out = model.context_encoder(latent_z, mask=inputs.attention_mask)
                decoder_input_embeds = model.projector(context_out)
                
                # Wrap embeddings for T5
                encoder_outputs_wrapped = BaseModelOutput(last_hidden_state=decoder_input_embeds)
                
                gen_ids = model.t5.generate(
                    encoder_outputs=encoder_outputs_wrapped, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            batch_preds = t5_tok.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        # Standard ChrF++ (6 char, 2 word)
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR DECODER: {TARGET_LANG}")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    # Save results
    results_df.to_csv(f"final_avg_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

Using device: cuda


You are using a model of type mt5 to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


Loading Model Weights...
Searching for pairs ending in ..._to_bn.csv

Testing Pair: ta -> bn


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 2.38 | ChrF++: 23.13 | BERT: 0.9047

Testing Pair: ml -> bn


...

   >> BLEU: 2.73 | ChrF++: 27.09 | BERT: 0.9141

Testing Pair: mr -> bn


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 3.33 | ChrF++: 26.63 | BERT: 0.9101

Testing Pair: as -> bn


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 4.55 | ChrF++: 29.81 | BERT: 0.9145

FINAL AGGREGATE RESULTS FOR DECODER: bn
Source Language  BLEU  ChrF++  IndicBERT
             ta  2.38   23.13       0.90
             ml  2.36   23.62       0.90
             or  3.22   28.92       0.91
             gu  2.73   27.09       0.91
             mr  3.33   26.63       0.91
             as  4.55   29.81       0.91
------------------------------------------------------------
 GLOBAL AVERAGE BLEU:       3.09
 GLOBAL AVERAGE ChrF++:     26.53
 GLOBAL AVERAGE IndicBERT:  0.9101


# With mt5-small (bengali)

In [25]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM
import torch.nn.functional as F

# --- CONFIGURATION ---
TARGET_LANG = "bn"  # The Decoder Language we are testing
# We use 'google/mt5-base' because standard 't5-base' does not support Indic scripts.
# If you have a fine-tuned T5 checkpoint, replace this string with your path (e.g., "/kaggle/input/my-finetuned-t5/")
BASELINE_MODEL_NAME = "google/mt5-small" 

# Evaluation Settings
BATCH_SIZE = 16

# --- HELPER: IndicBERTScore (Kept identical for fair comparison) ---
def calculate_indicbert_score(predictions, references, model_name='ai4bharat/indic-bert'):
    """Calculates semantic similarity using IndicBERT embeddings."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    eval_tokenizer = AutoTokenizer.from_pretrained(model_name)
    eval_model = AutoModel.from_pretrained(model_name).to(device)
    eval_model.eval()
    
    scores = []
    # Batch processing for speed
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        # Tokenize
        inputs = eval_tokenizer(batch_preds, batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.no_grad():
            # Get sentence embeddings (Mean Pooling)
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            # Normalize vectors
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            # Cosine Similarity
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP BASELINE MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading Baseline Model: {BASELINE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(BASELINE_MODEL_NAME).to(device)
model.eval()

# --- 2. FIND UNIQUE SOURCE FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found! Check your directory.")
    exit()

# --- 3. MAIN EVALUATION LOOP ---
results = []
processed_languages = set() 

for file_path in all_files:
    try:
        # A. Identify Source Language from Header
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue 
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue 
        
        processed_languages.add(src_col)
        
        print(f"\n[Baseline] Testing Pair: {src_col} -> {TARGET_LANG}")
        
        # B. Load and Sample Data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # C. Inference
        predictions = []
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            
            # Standard T5 Tokenization
            inputs = tokenizer(batch_src, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            
            with torch.no_grad():
                gen_ids = model.generate(
                    input_ids=inputs.input_ids, 
                    attention_mask=inputs.attention_mask,
                    max_length=64,
                    num_beams=4,
                    early_stopping=True
                )
            
            # Decode
            batch_preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            predictions.extend(batch_preds)
            
        # D. Calculate Metrics
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts)
        
        print(f"   >> BLEU: {bleu.score:.2f} | ChrF++: {chrf.score:.2f} | BERT: {bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL AGGREGATE RESULTS FOR BASELINE (mT5-small)")
    print("="*60)
    print(results_df.round(2).to_string(index=False))
    print("-" * 60)
    
    # Calculate Averages
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" BASELINE AVG BLEU:       {avg_bleu:.2f}")
    print(f" BASELINE AVG ChrF++:     {avg_chrf:.2f}")
    print(f" BASELINE AVG IndicBERT:  {avg_bert:.4f}")
    print("="*60)
    
    results_df.to_csv(f"baseline_results_{TARGET_LANG}.csv", index=False)
else:
    print("No results generated.")

Using device: cuda
Loading Baseline Model: google/mt5-small...


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Searching for pairs ending in ..._to_bn.csv

[Baseline] Testing Pair: ta -> bn


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 0.00 | ChrF++: 0.04 | BERT: 0.5093

[Baseline] Testing Pair: ml -> bn


...

   >> BLEU: 0.00 | ChrF++: 0.03 | BERT: 0.4876

[Baseline] Testing Pair: mr -> bn


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 0.00 | ChrF++: 0.04 | BERT: 0.5199

[Baseline] Testing Pair: as -> bn


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


   >> BLEU: 0.01 | ChrF++: 0.33 | BERT: 0.5313

FINAL AGGREGATE RESULTS FOR BASELINE (mT5-small)
Source Language  BLEU  ChrF++  IndicBERT
             ta  0.00    0.04       0.51
             ml  0.00    0.05       0.51
             or  0.00    0.07       0.54
             gu  0.00    0.03       0.49
             mr  0.00    0.04       0.52
             as  0.01    0.33       0.53
------------------------------------------------------------
 BASELINE AVG BLEU:       0.00
 BASELINE AVG ChrF++:     0.09
 BASELINE AVG IndicBERT:  0.5154


# IndicTrans2 (bengali)

In [26]:
import torch
import pandas as pd
import sacrebleu
import glob
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModel
from IndicTransToolkit import IndicProcessor
import torch.nn.functional as F
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Fetch the secret token
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Login automatically
login(token=hf_token, add_to_git_credential=False)

# --- CONFIGURATION ---
TARGET_LANG = "bn"  # ben_Beng
TARGET_LANG_CODE = "ben_Beng"

LANG_CODE_MAP = {
    "as": "asm_Beng", "bn": "ben_Beng", "hi": "hin_Deva", "en": "eng_Latn",
    "ta": "tam_Taml", "te": "tel_Telu", "mr": "mar_Deva", "gu": "guj_Gujr",
    "kn": "kan_Knda", "ml": "mal_Mlym", "pa": "pan_Guru", "or": "ory_Orya", "ur": "urd_Arab",
}

# SPEED OPTIMIZATION: Use smaller model for faster inference
# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"  # 1B params, slower
MODEL_NAME = "ai4bharat/indictrans2-indic-indic-dist-320M"  # 320M params, 3x faster

# MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"


BATCH_SIZE = 32  # Increased from 16

# --- HELPER: IndicBERTScore ---
def calculate_indicbert_score(predictions, references, eval_tokenizer, eval_model, device):
    scores = []
    for i in range(0, len(predictions), 32):
        batch_preds = predictions[i : i + 32]
        batch_refs = references[i : i + 32]
        
        with torch.no_grad():
            input_preds = eval_tokenizer(batch_preds, padding=True, truncation=True, return_tensors="pt").to(device)
            input_refs = eval_tokenizer(batch_refs, padding=True, truncation=True, return_tensors="pt").to(device)
            
            out_preds = eval_model(**input_preds).last_hidden_state.mean(dim=1)
            out_refs = eval_model(**input_refs).last_hidden_state.mean(dim=1)
            
            out_preds = F.normalize(out_preds, p=2, dim=1)
            out_refs = F.normalize(out_refs, p=2, dim=1)
            
            sims = (out_preds * out_refs).sum(dim=1)
            scores.extend(sims.cpu().tolist())

    return sum(scores) / len(scores) if scores else 0

# --- 1. SETUP MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Loading IndicTrans2 model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, 
    trust_remote_code=True,
    torch_dtype=torch.float16,  # FP16 for speed
).to(device)
model.eval()

ip = IndicProcessor(inference=True)
print("IndicTrans2 model loaded successfully!")

# NEW: Load IndicBERT globally so it doesn't reload for every language
print("Loading IndicBERT evaluation model...")
eval_tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-bert')
eval_model = AutoModel.from_pretrained('ai4bharat/indic-bert').to(device)
eval_model.eval()
print("All models loaded successfully!")

# --- 2. FIND FILES ---
print(f"Searching for pairs ending in ..._to_{TARGET_LANG}.csv")
file_pattern = f"*_to_{TARGET_LANG}.csv"
all_files = glob.glob(f"/kaggle/input/**/{file_pattern}", recursive=True)

if not all_files:
    print("No files found!")
    exit()

# --- 3. MAIN EVALUATION LOOP WITH PROGRESS BARS ---
results = []
processed_languages = set()

# Outer progress bar for languages
for file_path in tqdm(all_files, desc="Overall Progress", unit="language"):
    try:
        header_df = pd.read_csv(file_path, nrows=0)
        cols = header_df.columns.tolist()
        
        if TARGET_LANG not in cols:
            continue
            
        src_col = [c for c in cols if c != TARGET_LANG][0]
        
        if src_col in processed_languages:
            continue
        
        processed_languages.add(src_col)
        
        src_lang_code = LANG_CODE_MAP.get(src_col, src_col)
        tgt_lang_code = LANG_CODE_MAP.get(TARGET_LANG, TARGET_LANG)
        
        print(f"\n{'='*60}")
        print(f"Testing: {src_col} ({src_lang_code}) → {TARGET_LANG} ({tgt_lang_code})")
        print(f"{'='*60}")
        
        # Load data
        df = pd.read_csv(file_path, usecols=[src_col, TARGET_LANG])
        sample_df = df.dropna().astype(str)
        
        src_texts = sample_df[src_col].tolist()
        ref_texts = sample_df[TARGET_LANG].tolist()
        
        # Inference with inner progress bar
        predictions = []
        num_batches = (len(src_texts) + BATCH_SIZE - 1) // BATCH_SIZE
        
        print(f"Processing {len(src_texts)} sentences in {num_batches} batches...")
        for i in range(0, len(src_texts), BATCH_SIZE):
            batch_src = src_texts[i : i + BATCH_SIZE]
            print(f"  Batch {i//BATCH_SIZE + 1}/{num_batches}...", end="\r")
                    
            # Preprocess
            batch = ip.preprocess_batch(batch_src, src_lang=src_lang_code, tgt_lang=tgt_lang_code)
            
            # Tokenize
            inputs = tokenizer(
                batch,
                truncation=True,
                padding="longest",
                return_tensors="pt",
                return_attention_mask=True
            ).to(device)
            
            # Generate (use_cache=False to avoid bug, num_beams=1 for speed)
            with torch.no_grad():
                generated_tokens = model.generate(
                    **inputs,
                    use_cache=False,
                    min_length=0,
                    max_length=256,
                    num_beams=1,
                    num_return_sequences=1,
                )
            
            # Decode (Removed deprecated as_target_tokenizer warning)
            batch_preds = tokenizer.batch_decode(
                generated_tokens, 
                skip_special_tokens=True, 
                clean_up_tokenization_spaces=True
            )
            
            # Post-process
            batch_preds = ip.postprocess_batch(batch_preds, lang=tgt_lang_code)
            predictions.extend(batch_preds)
            
            # Clean up GPU memory
            del inputs, generated_tokens
            torch.cuda.empty_cache()
        
        # Calculate Metrics
        print(f"\nCalculating metrics for {src_col}→{TARGET_LANG}...")
        bleu = sacrebleu.corpus_bleu(predictions, [ref_texts])
        chrf = sacrebleu.corpus_chrf(predictions, [ref_texts], char_order=4, word_order=2, beta=2)
        bert = calculate_indicbert_score(predictions, ref_texts, eval_tokenizer, eval_model, device)
        
        print(f"✓ Results: BLEU={bleu.score:.2f} | ChrF++={chrf.score:.2f} | BERT={bert:.4f}")
        
        results.append({
            "Source Language": src_col,
            "Source Code": src_lang_code,
            "BLEU": bleu.score,
            "ChrF++": chrf.score,
            "IndicBERT": bert
        })
        
    except Exception as e:
        print(f"\n✗ Error processing {os.path.basename(file_path)}: {e}")
        import traceback
        traceback.print_exc()

# --- 4. FINAL REPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    print("\n" + "="*70)
    print(f"FINAL AGGREGATE RESULTS FOR TARGET: {TARGET_LANG} ({tgt_lang_code})")
    print("="*70)
    print(results_df.round(2).to_string(index=False))
    print("-" * 70)
    
    avg_bleu = results_df["BLEU"].mean()
    avg_chrf = results_df["ChrF++"].mean()
    avg_bert = results_df["IndicBERT"].mean()
    
    print(f" GLOBAL AVERAGE BLEU:       {avg_bleu:.2f}")
    print(f" GLOBAL AVERAGE ChrF++:     {avg_chrf:.2f}")
    print(f" GLOBAL AVERAGE IndicBERT:  {avg_bert:.4f}")
    print("="*70)
    
    results_df.to_csv(f"indictrans2_results_{TARGET_LANG}.csv", index=False)
    print(f"✓ Results saved to indictrans2_results_{TARGET_LANG}.csv")
else:
    print("No results generated.")

Using device: cuda
Loading IndicTrans2 model: ai4bharat/indictrans2-indic-indic-dist-320M
IndicTrans2 model loaded successfully!
Loading IndicBERT evaluation model...
All models loaded successfully!
Searching for pairs ending in ..._to_bn.csv


Overall Progress:   0%|          | 0/6 [00:00<?, ?language/s]


Testing: ta (tam_Taml) → bn (ben_Beng)
Processing 1024 sentences in 32 batches...
  Batch 32/32...
Calculating metrics for ta→bn...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Overall Progress:  17%|█▋        | 1/6 [05:25<27:07, 325.44s/language]

✓ Results: BLEU=9.21 | ChrF++=41.74 | BERT=0.9577

Testing: ml (mal_Mlym) → bn (ben_Beng)
Processing 1024 sentences in 32 batches...
  Batch 32/32...
Calculating metrics for ml→bn...


...

✓ Results: BLEU=9.89 | ChrF++=44.39 | BERT=0.9614

Testing: mr (mar_Deva) → bn (ben_Beng)
Processing 1024 sentences in 32 batches...
  Batch 32/32...
Calculating metrics for mr→bn...


Overall Progress:  83%|████████▎ | 5/6 [29:02<05:32, 332.33s/language]

✓ Results: BLEU=10.60 | ChrF++=44.41 | BERT=0.9604

Testing: as (asm_Beng) → bn (ben_Beng)
Processing 1024 sentences in 32 batches...
  Batch 32/32...
Calculating metrics for as→bn...


Overall Progress: 100%|██████████| 6/6 [39:25<00:00, 394.24s/language]

✓ Results: BLEU=10.25 | ChrF++=44.39 | BERT=0.9498

FINAL AGGREGATE RESULTS FOR TARGET: bn (ben_Beng)
Source Language Source Code  BLEU  ChrF++  IndicBERT
             ta    tam_Taml  9.21   41.74       0.96
             ml    mal_Mlym  9.94   43.35       0.96
             or    ory_Orya  9.46   44.75       0.95
             gu    guj_Gujr  9.89   44.39       0.96
             mr    mar_Deva 10.60   44.41       0.96
             as    asm_Beng 10.25   44.39       0.95
----------------------------------------------------------------------
 GLOBAL AVERAGE BLEU:       9.89
 GLOBAL AVERAGE ChrF++:     43.84
 GLOBAL AVERAGE IndicBERT:  0.9567
✓ Results saved to indictrans2_results_bn.csv
